<a href="https://colab.research.google.com/github/Yascaram/Datathon-Fase-5-/blob/main/Modelo_risco_tech5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# %load /content/modelo_risco.py
"""
Modelo Preditivo de Risco de Defasagem — Passos Mágicos (2022–2024)
====================================================================
Identifica padrões nos indicadores que antecedem quedas de desempenho
ou aumento de defasagem e atribui uma probabilidade de risco a cada aluno.

Definição de "em risco":
  - Queda de INDE > 0.3 pontos no ano seguinte, OU
  - Aumento da defasagem escolar (piora em ≥ 1 nível)

Modelo: Gradient Boosting Calibrado (AUC-ROC = 0.68, 5-fold CV)

Dependências: pandas, numpy, scipy, scikit-learn, openpyxl
  pip install pandas numpy scipy scikit-learn openpyxl
"""

import re
import warnings
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# 1. CARGA E LIMPEZA
# ══════════════════════════════════════════════════════════════════════════════

df22 = pd.read_excel("2022.xlsx")
df23 = pd.read_excel("2023.xlsx")
df24 = pd.read_excel("2024.xlsx")


def clean_2022(df):
    d = df.copy()
    d.rename(columns={
        "Idade 22": "Idade", "INDE 22": "INDE", "Pedra 22": "Pedra",
        "Matem": "Mat", "Portug": "Por", "Inglês": "Ing",
        "Defas": "Defasagem", "Fase ideal": "Fase Ideal",
    }, inplace=True)
    d["Ano"] = 2022
    d["Gênero"] = d["Gênero"].map({"Menina": "Feminino", "Menino": "Masculino"})
    d["Fase"] = d["Fase"].map({
        0: "ALFA", 1: "FASE 1", 2: "FASE 2", 3: "FASE 3", 4: "FASE 4",
        5: "FASE 5", 6: "FASE 6", 7: "FASE 7", 8: "FASE 8",
    })
    d["Instituição de ensino"] = d["Instituição de ensino"].replace(
        {"Escola Pública": "Pública", "Rede Decisão": "Privada", "Escola JP II": "Privada"}
    )
    return d


def clean_2023(df):
    d = df.copy()
    d.rename(columns={
        "Nome Anonimizado": "Nome", "INDE 2023": "INDE", "Pedra 2023": "Pedra",
    }, inplace=True)
    d["Ano"] = 2023
    d["Fase"] = d["Fase"].str.upper().str.strip()
    d["Pedra"] = d["Pedra"].replace({"Agata": "Ágata"})
    return d


def clean_2024(df):
    d = df.copy()
    d.rename(columns={
        "Nome Anonimizado": "Nome", "INDE 2024": "INDE", "Pedra 2024": "Pedra",
    }, inplace=True)
    d["Ano"] = 2024
    d["INDE"] = pd.to_numeric(d["INDE"], errors="coerce")

    def normalizar_fase(f):
        f = str(f).strip().upper()
        if f == "ALFA":
            return "ALFA"
        m = re.match(r"^(\d+)", f)
        if m:
            n = int(m.group(1))
            if 1 <= n <= 8:
                return f"FASE {n}"
        return f

    d["Fase"] = d["Fase"].apply(normalizar_fase)
    d["Pedra"] = d["Pedra"].replace({"Agata": "Ágata", "INCLUIR": np.nan})
    return d


df22c = clean_2022(df22)
df23c = clean_2023(df23)
df24c = clean_2024(df24)


# ══════════════════════════════════════════════════════════════════════════════
# 2. CONSTRUÇÃO DO DATASET LONGITUDINAL (TARGET)
# ══════════════════════════════════════════════════════════════════════════════

def build_pairs(df_t, df_t1, s1, s2):
    """
    Para cada aluno presente nos dois anos consecutivos:
      - Extrai indicadores do ano T (features)
      - Verifica se houve queda de INDE ou aumento de defasagem no ano T+1 (target)
    """
    feats = ["RA", "INDE", "IDA", "IEG", "IPS", "IAA", "IPV", "IAN",
             "Defasagem", "Mat", "Por"]
    a = (df_t[[c for c in feats if c in df_t.columns]]
         .add_suffix(s1).rename(columns={f"RA{s1}": "RA"}))
    b = (df_t1[[c for c in feats if c in df_t1.columns]]
         .add_suffix(s2).rename(columns={f"RA{s2}": "RA"}))
    paired = a.merge(b, on="RA")

    i1, i2 = f"INDE{s1}", f"INDE{s2}"
    d1, d2 = f"Defasagem{s1}", f"Defasagem{s2}"

    queda_inde = (
        (paired[i2] - paired[i1]) < -0.3
        if (i1 in paired.columns and i2 in paired.columns)
        else pd.Series(False, index=paired.index)
    )
    aumento_def = (
        (paired[d2] - paired[d1]) < -1
        if (d1 in paired.columns and d2 in paired.columns)
        else pd.Series(False, index=paired.index)
    )
    paired["em_risco"] = (queda_inde | aumento_def).astype(int)
    return paired


def normalize_pairs(df, suffix):
    """Renomeia colunas sufixadas para nomes padrão."""
    feats = ["INDE", "IDA", "IEG", "IPS", "IAA", "IPV", "IAN",
             "Defasagem", "Mat", "Por"]
    cols = [f"{f}{suffix}" for f in feats if f"{f}{suffix}" in df.columns]
    d = df[["RA"] + cols + ["em_risco"]].copy()
    d.columns = ["RA"] + [c.replace(suffix, "") for c in cols] + ["em_risco"]
    return d


p1 = build_pairs(df22c, df23c, "_22", "_23")
p2 = build_pairs(df23c, df24c, "_23", "_24")
all_data = pd.concat([normalize_pairs(p1, "_22"),
                      normalize_pairs(p2, "_23")], ignore_index=True)

n_total = len(all_data)
n_risco = all_data["em_risco"].sum()
print(f"Dataset de treino: {n_total} alunos-período | "
      f"Em risco: {n_risco} ({n_risco/n_total*100:.1f}%)")


# ══════════════════════════════════════════════════════════════════════════════
# 3. ENGENHARIA DE FEATURES
# ══════════════════════════════════════════════════════════════════════════════

def build_features(df):
    """
    Cria features derivadas que capturam padrões de risco identificados
    na análise longitudinal:

    superestimacao  : IAA muito acima do IDA → aluno desconhece sua real situação
    gap_ieg_ipv     : engajamento alto mas ponto de virada baixo → esforço sem transformação
    plateau         : INDE alto mas IAN baixo → bom desempenho geral com defasagem de nível
    ratio_ida_inde  : quão grande é a contribuição do aprendizado no índice geral
    ips_sobre_ieg   : razão bem-estar / engajamento → IPS caindo com IEG alto = insustentável
    """
    d = df.copy()
    d["superestimacao"] = d["IAA"] - d["IDA"]
    d["gap_ieg_ipv"]    = d["IEG"] - d["IPV"]
    d["plateau"]        = d["INDE"] - d["IAN"]
    d["ratio_ida_inde"] = d["IDA"] / (d["INDE"] + 0.01)
    d["ips_sobre_ieg"]  = d["IPS"] / (d["IEG"] + 0.01)
    return d


all_data = build_features(all_data)

FEATURES = [
    # Indicadores base
    "INDE", "IDA", "IEG", "IPS", "IAA", "IPV", "IAN",
    "Defasagem", "Mat", "Por",
    # Features derivadas de risco
    "superestimacao", "gap_ieg_ipv", "plateau",
    "ratio_ida_inde", "ips_sobre_ieg",
]

X = all_data[FEATURES]
y = all_data["em_risco"]


# ══════════════════════════════════════════════════════════════════════════════
# 4. AVALIAÇÃO DE MODELOS (5-fold Cross-Validation)
# ══════════════════════════════════════════════════════════════════════════════

print("\n=== Comparação de Modelos (AUC-ROC, 5-fold CV) ===")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    "Regressão Logística": LogisticRegression(
        max_iter=1000, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=6, random_state=42, class_weight="balanced"),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
}

for nome, clf in modelos.items():
    pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("clf", clf)])
    aucs = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
    print(f"  {nome:<25s}: AUC = {aucs.mean():.3f} ± {aucs.std():.3f}")


# ══════════════════════════════════════════════════════════════════════════════
# 5. MODELO FINAL — GRADIENT BOOSTING CALIBRADO
# ══════════════════════════════════════════════════════════════════════════════
# Calibração isotônica garante que a probabilidade prevista seja interpretável:
# prob=0.60 significa que ~60% dos alunos com esse perfil entraram em risco.

print("\n=== Métricas do Modelo Final (GBM + Calibração Isotônica) ===")

imp_fn   = SimpleImputer(strategy="median")
X_imp    = imp_fn.fit_transform(X)
gbm_base = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
model    = CalibratedClassifierCV(gbm_base, cv=5, method="isotonic")
model.fit(X_imp, y)

# Métricas out-of-fold (sem vazamento de dados)
cv_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("cal", CalibratedClassifierCV(
        GradientBoostingClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
        cv=5, method="isotonic")),
])
y_prob_cv = cross_val_predict(cv_pipe, X, y, cv=cv, method="predict_proba")[:, 1]
y_pred_cv = (y_prob_cv >= 0.45).astype(int)

print(f"  AUC-ROC       : {roc_auc_score(y, y_prob_cv):.3f}")
print(f"  Limiar adotado: 0.45")
print(classification_report(y, y_pred_cv, target_names=["Sem risco", "Em risco"]))


# ══════════════════════════════════════════════════════════════════════════════
# 6. IMPORTÂNCIA DAS FEATURES E PADRÕES DE RISCO
# ══════════════════════════════════════════════════════════════════════════════

# Treina GBM sem calibração apenas para extrair importâncias
gbm_raw = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gbm_raw.fit(X_imp, y)

imp_features = (
    pd.Series(gbm_raw.feature_importances_, index=FEATURES)
    .sort_values(ascending=False)
)
print("\n=== Importância das Features ===")
for feat, val in imp_features.items():
    bar = "█" * int(val * 60)
    print(f"  {feat:<20s}: {val:.4f}  {bar}")

print("\n=== Perfil Médio: Alunos que Entraram em Risco vs Não Entraram ===")
comparativo = all_data.groupby("em_risco")[
    ["INDE", "IDA", "IEG", "IPS", "IAA", "IPV", "IAN",
     "Defasagem", "superestimacao", "gap_ieg_ipv"]
].mean().round(2)
comparativo.index = ["Sem risco (0)", "Em risco (1)"]
print(comparativo.T.to_string())

print("""
Interpretação dos padrões:
  → Alunos em risco têm IAA mais alto que os seguros (+0.8 pts em média)
    — superestimam sua performance antes da queda
  → gap_ieg_ipv positivo: engajamento alto sem transformação real (ponto de virada baixo)
  → plateau: INDE geral bom mas com defasagem de nível acumulada
  → IPS cai enquanto IEG se mantém: sinal de esgotamento iminente
""")


# ══════════════════════════════════════════════════════════════════════════════
# 7. APLICAR O MODELO NOS ALUNOS DE 2024
# ══════════════════════════════════════════════════════════════════════════════

df24_pred = df24c[[
    "RA", "Fase", "Pedra", "INDE", "IDA", "IEG",
    "IPS", "IAA", "IPV", "IAN", "Defasagem", "Mat", "Por"
]].copy()
df24_pred = build_features(df24_pred)

X24     = imp_fn.transform(df24_pred[FEATURES])
probs24 = model.predict_proba(X24)[:, 1]
df24_pred["prob_risco"] = probs24
df24_pred["prob_risco_%"] = (probs24 * 100).round(1)


def classificar_risco(p):
    if p >= 0.55:
        return "Alto"
    if p >= 0.40:
        return "Médio"
    return "Baixo"


df24_pred["nivel_risco"] = df24_pred["prob_risco"].apply(classificar_risco)

print("=== Distribuição de Risco — Alunos 2024 ===")
print(df24_pred["nivel_risco"].value_counts().to_string())
print(f"\nProbabilidade média de risco: {probs24.mean()*100:.1f}%")

print("\nRisco médio por Fase:")
print(
    df24_pred[df24_pred["Fase"].str.startswith("FASE", na=False) | (df24_pred["Fase"] == "ALFA")]
    .groupby("Fase")["prob_risco_%"].mean().round(1).sort_values(ascending=False).to_string()
)

print("\nRisco médio por Pedra:")
print(df24_pred.groupby("Pedra")["prob_risco_%"].mean().round(1)
      .sort_values(ascending=False).to_string())

print("\nTop 20 alunos em maior risco:")
cols_show = ["RA", "Fase", "Pedra", "INDE", "IDA", "IEG",
             "IPV", "Defasagem", "superestimacao", "gap_ieg_ipv", "prob_risco_%", "nivel_risco"]
print(df24_pred.nlargest(20, "prob_risco")[cols_show].to_string(index=False))


# ══════════════════════════════════════════════════════════════════════════════
# 8. EXPORTAR PARA EXCEL COM FORMATAÇÃO
# ══════════════════════════════════════════════════════════════════════════════

# ── Preparar abas ─────────────────────────────────────────────────────────────
resumo_fase = (
    df24_pred[df24_pred["Fase"].isin(
        ["ALFA", "FASE 1", "FASE 2", "FASE 3", "FASE 4",
         "FASE 5", "FASE 6", "FASE 7", "FASE 8"])]
    .groupby("Fase")
    .agg(
        N=("RA", "count"),
        Risco_medio_pct=("prob_risco_%", "mean"),
        Alto_risco=("nivel_risco", lambda x: (x == "Alto").sum()),
        Medio_risco=("nivel_risco", lambda x: (x == "Médio").sum()),
        Baixo_risco=("nivel_risco", lambda x: (x == "Baixo").sum()),
        INDE_medio=("INDE", "mean"),
        IDA_medio=("IDA", "mean"),
        IEG_medio=("IEG", "mean"),
    )
    .round(2)
    .reset_index()
)

# Resultado completo (colunas selecionadas para o Excel)
resultado = df24_pred[[
    "RA", "Fase", "Pedra", "INDE", "IDA", "IEG", "IPS",
    "IAA", "IPV", "IAN", "Defasagem", "Mat", "Por",
    "superestimacao", "gap_ieg_ipv", "plateau",
    "prob_risco_%", "nivel_risco",
]].sort_values("prob_risco_%", ascending=False).copy()
resultado.columns = [
    "RA", "Fase", "Pedra", "INDE", "IDA", "IEG", "IPS",
    "IAA", "IPV", "IAN", "Defasagem", "Mat", "Por",
    "Superestimação (IAA-IDA)", "Gap IEG-IPV", "Plateau (INDE-IAN)",
    "Prob. Risco (%)", "Nível de Risco",
]

out_path = "risco_defasagem_2024.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    resultado.to_excel(writer, sheet_name="Previsão por Aluno", index=False)
    resumo_fase.to_excel(writer, sheet_name="Resumo por Fase", index=False)

# ── Formatação openpyxl ────────────────────────────────────────────────────────
wb = load_workbook(out_path)

DARK_BLUE  = "1F4E79"
LIGHT_BLUE = "D6E4F0"
WHITE      = "FFFFFF"
RED        = "C00000"
ORANGE     = "FF9900"
GREEN      = "375623"
LIGHT_RED  = "FFD7D7"
LIGHT_ORA  = "FFF2CC"
LIGHT_GRN  = "E2EFDA"

thin = Side(style="thin", color="B0B0B0")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)


def style_sheet(ws, col_widths: dict):
    """Aplica header escuro + zebra + bordas + freeze."""
    headers = [cell.value for cell in ws[1]]

    for col_idx, h in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill     = PatternFill("solid", start_color=DARK_BLUE)
        cell.font     = Font(name="Arial", bold=True, color=WHITE, size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border   = BORDER
        ws.column_dimensions[get_column_letter(col_idx)].width = col_widths.get(h, 13)
    ws.row_dimensions[1].height = 30

    for row_idx in range(2, ws.max_row + 1):
        fill = PatternFill("solid", start_color=LIGHT_BLUE) if row_idx % 2 == 0 else PatternFill()
        for col_idx, h in enumerate(headers, 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            if not cell.fill or cell.fill.fill_type in (None, "none"):
                cell.fill = fill
            cell.font   = Font(name="Arial", size=10)
            cell.border = BORDER
            if h in ("INDE", "IDA", "IEG", "IPS", "IAA", "IPV", "IAN",
                     "Mat", "Por", "Prob. Risco (%)", "Risco_medio_pct",
                     "Superestimação (IAA-IDA)", "Gap IEG-IPV", "Plateau (INDE-IAN)"):
                cell.alignment  = Alignment(horizontal="center")
                if isinstance(cell.value, (int, float)):
                    cell.number_format = "0.0"
            elif h in ("Defasagem", "N", "Alto_risco", "Medio_risco", "Baixo_risco"):
                cell.alignment  = Alignment(horizontal="center")
            else:
                cell.alignment  = Alignment(horizontal="left")

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions


# Colorir coluna "Nível de Risco"
def colorir_nivel_risco(ws, col_name):
    headers = [cell.value for cell in ws[1]]
    if col_name not in headers:
        return
    col_idx = headers.index(col_name) + 1
    cores   = {"Alto": (RED, WHITE), "Médio": (ORANGE, "000000"), "Baixo": (GREEN, WHITE)}
    for row in range(2, ws.max_row + 1):
        cell = ws.cell(row=row, column=col_idx)
        if cell.value in cores:
            bg, fg = cores[cell.value]
            cell.fill = PatternFill("solid", start_color=bg)
            cell.font = Font(name="Arial", bold=True, color=fg, size=10)
            cell.alignment = Alignment(horizontal="center")


# ── Aba 1: Previsão por Aluno ─────────────────────────────────────────────────
ws1 = wb["Previsão por Aluno"]
style_sheet(ws1, {
    "RA": 10, "Fase": 10, "Pedra": 11, "INDE": 8, "IDA": 8, "IEG": 8,
    "IPS": 8, "IAA": 8, "IPV": 8, "IAN": 8, "Defasagem": 11,
    "Mat": 7, "Por": 7,
    "Superestimação (IAA-IDA)": 20, "Gap IEG-IPV": 13, "Plateau (INDE-IAN)": 17,
    "Prob. Risco (%)": 14, "Nível de Risco": 14,
})
colorir_nivel_risco(ws1, "Nível de Risco")

# Escala de cor na coluna Prob. Risco (%)
headers1 = [c.value for c in ws1[1]]
prob_col = get_column_letter(headers1.index("Prob. Risco (%)") + 1)
ws1.conditional_formatting.add(
    f"{prob_col}2:{prob_col}{ws1.max_row}",
    ColorScaleRule(
        start_type="min", start_color="63BE7B",
        mid_type="percentile", mid_value=50, mid_color="FFEB84",
        end_type="max", end_color="F8696B",
    ),
)

# ── Aba 2: Resumo por Fase ────────────────────────────────────────────────────
ws2 = wb["Resumo por Fase"]
style_sheet(ws2, {
    "Fase": 10, "N": 6, "Risco_medio_pct": 15, "Alto_risco": 12,
    "Medio_risco": 12, "Baixo_risco": 12,
    "INDE_medio": 12, "IDA_medio": 11, "IEG_medio": 11,
})
# Colorir células Alto_risco > 0
headers2 = [c.value for c in ws2[1]]
if "Alto_risco" in headers2:
    ac = get_column_letter(headers2.index("Alto_risco") + 1)
    for row in range(2, ws2.max_row + 1):
        cell = ws2.cell(row=row, column=headers2.index("Alto_risco") + 1)
        if isinstance(cell.value, (int, float)) and cell.value > 0:
            cell.fill = PatternFill("solid", start_color=LIGHT_RED)
            cell.font = Font(name="Arial", bold=True, color=RED, size=10)

wb.save(out_path)
print(f"\n✅ Arquivo salvo: {out_path}")
print(f"   Aba 1 — Previsão por Aluno : {len(resultado)} linhas")
print(f"   Aba 2 — Resumo por Fase    : {len(resumo_fase)} linhas")


Dataset de treino: 1365 alunos-período | Em risco: 430 (31.5%)

=== Comparação de Modelos (AUC-ROC, 5-fold CV) ===
  Regressão Logística      : AUC = 0.668 ± 0.008
  Random Forest            : AUC = 0.676 ± 0.029
  Gradient Boosting        : AUC = 0.681 ± 0.026

=== Métricas do Modelo Final (GBM + Calibração Isotônica) ===
  AUC-ROC       : 0.674
  Limiar adotado: 0.45
              precision    recall  f1-score   support

   Sem risco       0.71      0.93      0.81       935
    Em risco       0.55      0.18      0.27       430

    accuracy                           0.69      1365
   macro avg       0.63      0.55      0.54      1365
weighted avg       0.66      0.69      0.64      1365


=== Importância das Features ===
  gap_ieg_ipv         : 0.1346  ████████
  INDE                : 0.1112  ██████
  Por                 : 0.1041  ██████
  ratio_ida_inde      : 0.0983  █████
  plateau             : 0.0919  █████
  IAA                 : 0.0802  ████
  IPV                 : 0.0690  ███